In [46]:
import os 
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pertpy as pt
import scanpy as sc
from pathlib import Path 

plt.rcParams["figure.figsize"] = (3, 3)

In [47]:
def assign_synth_samples(adata):
    adata.obs["sample_id"] = None
    n_control, n_trt = np.unique(adata.obs["treatment"], return_counts=True)[1]
    ctr_samples = np.random.randint(0, 2, n_control)
    trt_samples = np.random.randint(2, 4, n_trt)
    
    adata.obs.loc[adata.obs.treatment==0, "sample_id"] = ctr_samples
    adata.obs.loc[adata.obs.treatment==1, "sample_id"] = trt_samples
    
    adata.obs["sample_id"] = adata.obs["sample_id"].astype("category")

In [48]:
seeds = [8, 42, 100]

In [49]:
data_path = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/data/pbmc68k_oversampled/")
result_folder = Path("/home/icb/alessandro.palma/environment/scFM_density_estimation/project_folder/results/abundance_test_experiment/milo/")

In [54]:
for i, seed in enumerate(seeds):
    np.random.seed(seed)
    for adata_name in os.listdir(data_path):
        result_name = f"oversamp_{adata_name.split('_')[1].replace('.h5ad', '')}_{i}"
        # Read AnnData 
        adata = sc.read_h5ad(data_path / adata_name)
        # Assign synthetic replicates 
        assign_synth_samples(adata)
        # Assign string treatment labels
        adata.obs["treatment_label"] = adata.obs.treatment.astype(str)
        # Initialize Milo
        milo = pt.tl.Milo()
        mdata = milo.load(adata)
        # Create neighbors
        milo.make_nhoods(mdata["rna"], prop=0.1)
        # Count per sample  
        mdata = milo.count_nhoods(mdata, sample_col="sample_id")
        # Differential abundance analysis 
        milo.da_nhoods(mdata, design="~treatment_label", solver='pydeseq2')
        # Build neighbor graph 
        milo.build_nhood_graph(mdata)
        adata = mdata["rna"][mdata["milo"].var.index_cell]
        adata.obs["log_ratio"] = mdata["milo"].var.logFC.to_numpy()
        
        # Compute log-ratio
        # nhoods = mdata["rna"].obsm["nhoods"]
        # logFC = mdata["milo"].var["logFC"].values[:, None]
        # per_cell_fc = nhoods.dot(logFC) / nhoods.sum(1)
        # adata.obs["log_ratio"] = per_cell_fc

        # del adata.uns["nhood_neighbors_key"]
        # adata = adata[mdata["milo"].var.index_cell]
        adata.write_h5ad(result_folder / (result_name + ".h5ad"))
        print("Results saved")

! Using X_pca as default embedding
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 2.54 seconds.

Fitting dispersion trend curve...
/home/icb/alessandro.palma/miniconda3/envs/scvi_env/lib/python3.11/site-packages/pydeseq2/dds.py:820: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.08 seconds.

/home/icb/alessandro.palma/miniconda3/envs/scvi_env/lib/python3.11/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 3.72 seconds.

Fitting LFCs...
... done in 2.42 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 1.15 seconds.

/tmp/ipykernel_722209/579558848.py:23: Implici

Log2 fold change & Wald test p-value: treatment_label 1 vs 0
       baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
0     13.732521        0.030123  0.717532  0.041982  0.966513  0.999619
1      7.772558        2.307688  0.935725  2.466202  0.013655  0.069272
2      5.274711       -1.177094  0.912511 -1.289950  0.197068  0.734489
3      9.502694        5.144482  1.589588  3.236363  0.001211  0.011962
4     12.820640       -0.290129  0.734685 -0.394903  0.692914  0.999619
...         ...             ...       ...       ...       ...       ...
4728  15.314288        4.822968  1.205727  4.000050  0.000063  0.003485
4729  11.345989       -0.173291  0.755871 -0.229260  0.818667  0.999619
4730  17.952483        3.385919  0.876801  3.861671  0.000113  0.003722
4731  16.500709       -0.131302  0.701992 -0.187042  0.851628  0.999619
4732   8.269726        0.518936  0.815110  0.636645  0.524356  0.999619

[4733 rows x 6 columns]
Results saved
! Using X_pca as default embedding


Fitting size factors...
... done in 0.00 seconds.



Using None as control genes, passed at DeseqDataSet initialization


...

Log2 fold change & Wald test p-value: treatment_label 1 vs 0
       baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
0     13.771153       -0.113348  0.736911 -0.153816  0.877755  0.992615
1      8.112179        0.250908  0.814398  0.308091  0.758013  0.992615
2      5.153977        0.689593  0.935372  0.737240  0.460976  0.992615
3     10.071851       -0.079738  0.781151 -0.102078  0.918695  0.992615
4     12.789272       -0.688215  0.745970 -0.922577  0.356228  0.992615
...         ...             ...       ...       ...       ...       ...
4728  16.226442       -0.010653  0.706593 -0.015077  0.987971  0.992615
4729  11.304147        0.626301  0.757382  0.826929  0.408277  0.992615
4730  18.949641       -0.503455  0.693702 -0.725751  0.467991  0.992615
4731  16.475012        0.290969  0.709464  0.410125  0.681714  0.992615
4732   8.376873       -0.887144  0.815787 -1.087470  0.276829  0.992615

[4733 rows x 6 columns]
Results saved
! Using X_pca as default embedding


Fitting size factors...
... done in 0.00 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 2.34 seconds.

Fitting dispersion trend curve...
/home/icb/alessandro.palma/miniconda3/envs/scvi_env/lib/python3.11/site-packages/pydeseq2/dds.py:820: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.10 seconds.

/home/icb/alessandro.palma/miniconda3/envs/scvi_env/lib/python3.11/site-packages/pydeseq2/dds.py:548: UserWarning: As the residual degrees of freedom is less than 3, the distribution of log dispersions is especially asymmetric and likely to be poorly estimated by the MAD.
  self.fit_dispersion_prior()
Fitting MAP dispersions...
... done in 3.62 seconds.

Fitting LFCs...
... done in 2.31 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 1.12 seconds.

/tmp/ipykernel_722209/579558848.py:23: ImplicitModificationWarning: Trying to modify attribute `.

Log2 fold change & Wald test p-value: treatment_label 1 vs 0
       baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
0     13.741687        0.030319  0.753600  0.040232  0.967908  0.999923
1      8.066354        0.547898  0.830195  0.659962  0.509278  0.999923
2      5.214180       -1.071776  0.966594 -1.108817  0.267509  0.999923
3     10.027258        0.427353  0.791535  0.539904  0.589263  0.999923
4     12.810217       -0.186316  0.764658 -0.243659  0.807495  0.999923
...         ...             ...       ...       ...       ...       ...
4728  16.166603        0.369292  0.737355  0.500834  0.616488  0.999923
4729  11.354113       -0.839337  0.792959 -1.058488  0.289833  0.999923
4730  18.853773        0.341869  0.716983  0.476817  0.633493  0.999923
4731  16.581097       -0.820478  0.738253 -1.111378  0.266406  0.999923
4732   8.326243        0.271297  0.821468  0.330259  0.741204  0.999923

[4733 rows x 6 columns]
Results saved


In [55]:
mdata["milo"].var.logFC

0       0.030319
1       0.547898
2      -1.071776
3       0.427353
4      -0.186316
          ...   
4728    0.369292
4729   -0.839337
4730    0.341869
4731   -0.820478
4732    0.271297
Name: logFC, Length: 4733, dtype: float64